In [1]:
import pandas as pd
import requests
import numpy as np
import time
import re

# load Excel and clean contaminants
df = pd.read_excel("1. 95p_Original_35277_LFQ_PD_WT_vs_Ts65Dn_4Mo-(1).xlsx")
df = df[~df['Accession'].str.startswith("Cont_")]
ratio_col = "Abundance Ratio: (Ts65Dn) / (WT)"

# utility functions
def clean_description(desc):
    return re.sub(r'\[.*?\]', '', desc).strip()
# species_prefix="mmu" for mouse
def get_kegg_gene_id(gene_symbol, species_prefix="mmu"):
    url = f"http://rest.kegg.jp/find/genes/{gene_symbol}"
    r = requests.get(url)
    if r.status_code != 200 or not r.text.strip():
        return None
    gene_symbol_lower = gene_symbol.lower()
    for line in r.text.strip().splitlines():
        if line.startswith(f"{species_prefix}:"):
            parts = line.split("\t")
            if len(parts) < 2:
                continue
            gene_id = parts[0]
            names = parts[1].lower().replace(";", ",").split(",")
            names = [n.strip() for n in names]
            if gene_symbol_lower in names:
                return gene_id
    return None
def get_kegg_pathway_ids(kegg_gene_id):
    url = f"http://rest.kegg.jp/link/pathway/{kegg_gene_id}"
    r = requests.get(url)
    if r.status_code != 200 or not r.text.strip():
        return []
    return [line.split("\t")[1].replace("path:", "")
            for line in r.text.strip().splitlines()]
def get_kegg_pathway_names(pathway_ids):
    names = []
    for pid in pathway_ids:
        url = f"http://rest.kegg.jp/get/{pid}"
        r = requests.get(url)
        if r.status_code == 200:
            for line in r.text.splitlines():
                if line.startswith("NAME"):
                    names.append(line.replace("NAME", "").strip())
                    break
        time.sleep(0.15)
    return names
def compute_hit_score(ratio):
    if pd.isna(ratio):
        return np.nan, None
    if ratio == 0:
        return 10, "Normal-only"
    if ratio == 1:
        return 0, "No-change"
    L = abs(np.log2(ratio))
    L = min(L, 10)
    return L, ("Up" if ratio > 1 else "Down")
# uniprot organism ID → 10090 (mouse)
def get_gene_symbol_uniprot(protein_name):
    query = protein_name + " AND organism:10090"
    url = f"https://www.uniprot.org/uniprotkb/search?query={query}&format=tsv&fields=genes"
    try:
        r = requests.get(url)
        if r.status_code == 200 and r.text.strip():
            lines = r.text.strip().splitlines()
            if len(lines) > 1:
                return lines[1].split()[0]
    except:
        pass
    return None

# containers for results
pathway_scores = {}
pathway_members = {}
hit_scores = []
updown_categories = []
protein_pathways = []

# main loop
for idx, row in df.iterrows():
    protein_name = clean_description(row["Description"])
    ratio = row[ratio_col]
    gene_symbol_full = row["Gene Symbol"] if not pd.isna(row["Gene Symbol"]) else ""
    gene_symbol = gene_symbol_full.split(";")[0].strip() if gene_symbol_full else get_gene_symbol_uniprot(protein_name)
    if not gene_symbol:
        print(f"{protein_name}: No gene symbol found")
        hit_scores.append(np.nan)
        updown_categories.append(None)
        protein_pathways.append("")
        continue
    # explicitly using mouse KEGG
    kegg_gene_id = get_kegg_gene_id(gene_symbol, species_prefix="mmu")
    if not kegg_gene_id:
        print(f"{protein_name} ({gene_symbol}): No KEGG gene ID found")
        hit_scores.append(np.nan)
        updown_categories.append(None)
        protein_pathways.append("")
        continue
    pathway_ids = get_kegg_pathway_ids(kegg_gene_id)
    if not pathway_ids:
        print(f"{protein_name} ({gene_symbol}, {kegg_gene_id}): No pathways found")
        hit_scores.append(np.nan)
        updown_categories.append(None)
        protein_pathways.append("")
        continue
    pathway_names = get_kegg_pathway_names(pathway_ids)
    protein_pathways.append("; ".join(pathway_names))
    hit_score, category = compute_hit_score(ratio)
    hit_scores.append(hit_score)
    updown_categories.append(category)
    for pid, pname in zip(pathway_ids, pathway_names):
        if pid not in pathway_scores:
            pathway_scores[pid] = {"name": pname, "total_score": 0, "protein_count": 0}
        pathway_scores[pid]["total_score"] += hit_score
        pathway_scores[pid]["protein_count"] += 1
        if pid not in pathway_members:
            pathway_members[pid] = []
        pathway_members[pid].append(protein_name)
    time.sleep(0.3)
    
# save outputs
df["Hit Score"] = hit_scores
df["Up/Down Category"] = updown_categories
df["Pathways"] = protein_pathways
pw_summary = pd.DataFrame([
    {
        "Pathway ID": pid,
        "Pathway Name": info["name"],
        "Total Hit Score": info["total_score"],
        "Protein Count": info["protein_count"]
    }
    for pid, info in pathway_scores.items()
]).sort_values("Total Hit Score", ascending=False)
pw_members_df = pd.DataFrame([
    {"Pathway ID": pid, "Pathway Name": pathway_scores[pid]["name"], "Proteins": "; ".join(prots)}
    for pid, prots in pathway_members.items()
])
output_filename = "2. 95p_Original_35277_LFQ_PD_WT_vs_Ts65Dn_4Mo-(1).xlsx"
with pd.ExcelWriter(output_filename, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Protein Data 4M", index=False)
    pw_summary.to_excel(writer, sheet_name="Pathway Summary 4M", index=False)
    pw_members_df.to_excel(writer, sheet_name="Pathway Members 4M", index=False)
print(f"\nSaved full mouse analysis to {output_filename}!")

Serum paraoxonase/arylesterase 2 (Pon2, mmu:330260): No pathways found
Phosphoglucomutase-like protein 5 (Pgm5, mmu:226041): No pathways found
Serine/threonine-protein kinase SBK2 (Sbk2, mmu:381836): No pathways found
Nucleolysin TIAR (Tial1, mmu:21843): No pathways found
Serum paraoxonase/arylesterase 1 (Pon1, mmu:18979): No pathways found
N-alpha-acetyltransferase 10 (Naa10, mmu:56292): No pathways found
eIF5-mimic protein 1 (Bzw2, mmu:66912): No pathways found
Serine/threonine-protein kinase Nek9 (Nek9, mmu:217718): No pathways found
Protein 4.2 (Epb42, mmu:13828): No pathways found
Alanyl-tRNA editing protein Aarsd1 (Aarsd1, mmu:69684): No pathways found
Immunoglobulin heavy constant gamma 2A (Ighg2a): No KEGG gene ID found
Prothrombin (F2): No KEGG gene ID found
Histone H1.0 (H1-0, mmu:14958): No pathways found
Translation factor Guf1, mitochondrial (Guf1, mmu:231279): No pathways found
Ig gamma-2A chain C region secreted form: No gene symbol found
AFG1-like ATPase (Afg1l, mmu:215

In [2]:
import pandas as pd
import requests
import time

# input file information
input_file = "2. 95p_Original_35277_LFQ_PD_WT_vs_Ts65Dn_4Mo-(1).xlsx"
sheet_name = "Pathway Summary 4M"
# load the sheet containing pathways
df = pd.read_excel(input_file, sheet_name=sheet_name)
# new columns to fill
categories = []
subcategories = []
def get_kegg_class(pathway_id):
    """Return (category, subcategory) from KEGG CLASS field."""
    url = f"http://rest.kegg.jp/get/{pathway_id}"
    r = requests.get(url)
    if r.status_code != 200:
        return None, None
    category = None
    subcategory = None
    for line in r.text.splitlines():
        if line.startswith("CLASS"):
            # Remove "CLASS" and split the remaining text
            class_text = line.replace("CLASS", "").strip()
            if ";" in class_text:
                parts = [x.strip() for x in class_text.split(";")]
                if len(parts) >= 2:
                    category = parts[0]
                    subcategory = parts[1]
                else:
                    # only one piece found
                    category = parts[0]
                    subcategory = None
            else:
                category = class_text
                subcategory = None
            break
    return category, subcategory

# process each pathway ID
for pid in df["Pathway ID"]:
    cat, subcat = get_kegg_class(pid)
    categories.append(cat)
    subcategories.append(subcat)
    time.sleep(0.2)  # KEGG rate limit safety
# add new columns
df["Category"] = categories
df["Subcategory"] = subcategories

# save output
output_file = "3. 95p_Original_35277_LFQ_PD_WT_vs_Ts65Dn_4Mo-(1).xlsx"
df.to_excel(output_file, index=False)
print(f"\nSaved pathway classes to: {output_file}")


Saved pathway classes to: 3. 95p_Original_35277_LFQ_PD_WT_vs_Ts65Dn_4Mo-(1).xlsx
